# FAN 리뷰 대응 실험 노트북 (신규)

- 목적: FAN vs MLP(Conv1x1 FFN) 효과, 비용-효익, 주기성 정합성 근거를 같은 파일에서 재현
- 기본 비교: `Dual_FreqDAE` vs `Transformer_DAE`
- 주 데이터: `evaluation_results_simemg/*.csv` (모델별 precomputed 결과)

In [ ]:
# Keep legacy data and import paths relative to the project root.
from pathlib import Path as _ProjectPath
import os as _project_os
import sys as _project_sys

_project_start = _ProjectPath.cwd().resolve()
_project_root = next(
    (_p for _p in (_project_start, *_project_start.parents)
     if (_p / "deepFilter" / "dl_models.py").is_file()
     and (_p / "Data_Preparation").is_dir()
     and (_p / "notebooks").is_dir()),
    None,
)
if _project_root is None:
    raise RuntimeError("Open this notebook from within the ecg_denoise project.")
_project_os.chdir(_project_root)
if str(_project_root) not in _project_sys.path:
    _project_sys.path.insert(0, str(_project_root))


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import pickle
from scipy.signal import welch

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('talk')

REPO = Path('.').resolve()
CSV_SEG = REPO / 'evaluation_results_simemg' / 'simemg_0221_fixed_segment_metrics.csv'
CSV_SEQ = REPO / 'evaluation_results_simemg' / 'simemg_0221_fixed_10s_sequence_metrics.csv'
W_DUAL = REPO / '0221_FIXED' / 'Dual_FreqDAE_weights.best.weights.h5'
W_TR = REPO / '0221_FIXED' / 'Transformer_DAE_weights.best.weights.h5'
P_DUAL = REPO / '0221_FIXED' / 'test_results_Dual_FreqDAE.pkl'
P_TR = REPO / '0221_FIXED' / 'test_results_Transformer_DAE.pkl'

print('REPO:', REPO)
print('CSV_SEG exists:', CSV_SEG.exists())
print('CSV_SEQ exists:', CSV_SEQ.exists())
print('W_DUAL exists:', W_DUAL.exists())
print('W_TR exists:', W_TR.exists())


## 1) FAN vs MLP 성능 비교 (paired segment-level)

In [ ]:

df = pd.read_csv(CSV_SEG)
df['snri_db'] = df['snr_denoised_db'] - df['snr_noisy_db']

models_target = ['Dual_FreqDAE', 'Transformer_DAE']
sub = df[df['model'].isin(models_target)].copy()
print('rows:', len(sub), 'models:', sub['model'].unique())

overall = sub.groupby('model')[['rmse_denoised','snr_denoised_db','snri_db','cos_denoised']].mean()
display(overall.sort_values('snr_denoised_db', ascending=False))

key = ['subject','session','modality','path','seg_idx']
tr = sub[sub.model=='Transformer_DAE'][key + ['rmse_denoised','snr_denoised_db','snri_db','cos_denoised']].rename(columns={
    'rmse_denoised':'rmse_tr','snr_denoised_db':'snr_tr','snri_db':'snri_tr','cos_denoised':'cos_tr'
})
du = sub[sub.model=='Dual_FreqDAE'][key + ['rmse_denoised','snr_denoised_db','snri_db','cos_denoised']].rename(columns={
    'rmse_denoised':'rmse_du','snr_denoised_db':'snr_du','snri_db':'snri_du','cos_denoised':'cos_du'
})
paired = tr.merge(du, on=key, how='inner')
print('paired N=', len(paired))

paired['d_rmse'] = paired['rmse_du'] - paired['rmse_tr']
paired['d_snr'] = paired['snr_du'] - paired['snr_tr']
paired['d_snri'] = paired['snri_du'] - paired['snri_tr']
paired['d_cos'] = paired['cos_du'] - paired['cos_tr']

summary = pd.Series({
    'mean_d_rmse(dual-tr)': paired['d_rmse'].mean(),
    'dual_better_rmse_frac': (paired['d_rmse'] < 0).mean(),
    'mean_d_snr_db(dual-tr)': paired['d_snr'].mean(),
    'dual_better_snr_frac': (paired['d_snr'] > 0).mean(),
    'mean_d_snri_db(dual-tr)': paired['d_snri'].mean(),
    'dual_better_snri_frac': (paired['d_snri'] > 0).mean(),
    'mean_d_cos(dual-tr)': paired['d_cos'].mean(),
    'dual_better_cos_frac': (paired['d_cos'] > 0).mean(),
})
display(summary.to_frame('value'))


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(paired['d_rmse'], bins=50, kde=True, ax=axes[0], color='tab:blue')
axes[0].axvline(0, color='k', ls='--')
axes[0].set_title('ΔRMSE (Dual-Transformer)')

sns.histplot(paired['d_snri'], bins=50, kde=True, ax=axes[1], color='tab:green')
axes[1].axvline(0, color='k', ls='--')
axes[1].set_title('ΔSNRi dB (Dual-Transformer)')

sns.histplot(paired['d_cos'], bins=50, kde=True, ax=axes[2], color='tab:orange')
axes[2].axvline(0, color='k', ls='--')
axes[2].set_title('ΔCosine (Dual-Transformer)')

plt.tight_layout(); plt.show()


## 2) 모달리티/피험자별 일관성

In [ ]:

by_mod = sub.groupby(['model','modality'])[['rmse_denoised','snri_db','cos_denoised']].mean().reset_index()
display(by_mod)

subj_tr = sub[sub.model=='Transformer_DAE'].groupby(['subject','modality'])[['rmse_denoised','snri_db']].mean().reset_index()
subj_du = sub[sub.model=='Dual_FreqDAE'].groupby(['subject','modality'])[['rmse_denoised','snri_db']].mean().reset_index()
subj = subj_tr.merge(subj_du, on=['subject','modality'], suffixes=('_tr','_du'))
subj['d_snri'] = subj['snri_db_du'] - subj['snri_db_tr']

pv = subj.pivot(index='subject', columns='modality', values='d_snri')
plt.figure(figsize=(9,6))
sns.heatmap(pv, cmap='coolwarm', center=0, annot=True, fmt='.2f')
plt.title('Subject×Modality ΔSNRi (Dual-Transformer)')
plt.show()


## 3) 비용-효익: 파라미터 수 비교 (`*.weights.h5` 기준)

In [ ]:

def count_h5_params(path: Path):
    total = 0
    with h5py.File(path, 'r') as f:
        def visitor(name, obj):
            nonlocal total
            if isinstance(obj, h5py.Dataset):
                total += int(np.prod(obj.shape))
        f.visititems(visitor)
    return total

n_dual = count_h5_params(W_DUAL)
n_tr = count_h5_params(W_TR)

cost_df = pd.DataFrame({
    'model': ['Dual_FreqDAE(FAN)','Transformer_DAE(MLP/Conv1x1 FFN)'],
    'params_from_h5': [n_dual, n_tr],
    'ratio_vs_transformer': [n_dual/n_tr, 1.0],
})
display(cost_df)


## 4) FAN harmonic branch 프록시 분석 (`input_linear_p` 가중치)

In [ ]:

fan_kernels = []
fan_names = []
with h5py.File(W_DUAL, 'r') as f:
    def visitor(name, obj):
        if isinstance(obj, h5py.Dataset) and 'input_linear_p/vars/0' in name:
            fan_names.append(name)
            fan_kernels.append(obj[...])
    f.visititems(visitor)

print('FAN p-kernels found:', len(fan_kernels))
rows = []
for nm, W in zip(fan_names, fan_kernels):
    col_norm = np.linalg.norm(W, axis=0)
    s = np.linalg.svd(W, compute_uv=False)
    p = s/(s.sum()+1e-12)
    ent = -(p*np.log(p+1e-12)).sum()
    rows.append({
        'layer': nm, 'in_dim': W.shape[0], 'p_dim': W.shape[1],
        'mean_col_norm': col_norm.mean(),
        'std_col_norm': col_norm.std(),
        'top_sv_ratio': s[0]/(s.sum()+1e-12),
        'sv_entropy': ent,
    })
fan_stats = pd.DataFrame(rows)
display(fan_stats.head())

fig, axes = plt.subplots(1,2, figsize=(14,4))
sns.boxplot(data=fan_stats, y='mean_col_norm', ax=axes[0])
axes[0].set_title('mean column norm across FAN p-layers')
sns.boxplot(data=fan_stats, y='top_sv_ratio', ax=axes[1])
axes[1].set_title('top singular value ratio across FAN p-layers')
plt.tight_layout(); plt.show()


## 5) (옵션) 생리학적 대역 정합성 분석
- pkl 예측이 모델 간 동일한 경우 dual-vs-transformer 비교는 건너뜁니다.

In [ ]:

X_du, P_du, Y_du = pickle.load(open(P_DUAL, 'rb'))
X_tr, P_tr, Y_tr = pickle.load(open(P_TR, 'rb'))

same_pred = np.allclose(P_du, P_tr)
print('Dual vs Transformer predictions identical?', same_pred)

fs = 500
bands = {
    'P/T_low_0.5_10': (0.5, 10),
    'QRS_mid_10_40': (10, 40),
    'EMG_high_40_100': (40, 100),
}

def band_power(sig):
    sig = sig.squeeze(-1)
    f, Pxx = welch(sig, fs=fs, nperseg=256, axis=1)
    out = {}
    for k, (lo, hi) in bands.items():
        m = (f >= lo) & (f < hi)
        out[k] = np.trapz(Pxx[:, m], f[m], axis=1)
    return out

bp_clean = band_power(Y_du)
bp_noisy = band_power(X_du)
bp_du = band_power(P_du)

rows = []
for b in bands:
    err_noisy = np.mean(np.abs(bp_noisy[b]-bp_clean[b])/(bp_clean[b]+1e-12))
    err_du = np.mean(np.abs(bp_du[b]-bp_clean[b])/(bp_clean[b]+1e-12))
    rows.append({'band': b, 'rel_err_noisy': err_noisy, 'rel_err_dual': err_du, 'improvement': err_noisy-err_du})

display(pd.DataFrame(rows))

if same_pred:
    print('[INFO] current pkl outputs are identical -> dual-vs-transformer band gap not estimable from these pkl files.')
else:
    bp_tr = band_power(P_tr)
    rows2 = []
    for b in bands:
        err_du = np.mean(np.abs(bp_du[b]-bp_clean[b])/(bp_clean[b]+1e-12))
        err_tr = np.mean(np.abs(bp_tr[b]-bp_clean[b])/(bp_clean[b]+1e-12))
        rows2.append({'band': b, 'dual_minus_transformer_error': err_du-err_tr})
    display(pd.DataFrame(rows2))


## 6) rebuttal 문구 자동 생성 (초안)

In [ ]:

mean_d_rmse = float(paired['d_rmse'].mean())
mean_d_snri = float(paired['d_snri'].mean())
frac_rmse = float((paired['d_rmse'] < 0).mean())
frac_snri = float((paired['d_snri'] > 0).mean())
param_ratio = float(n_dual / n_tr)

lines = [
    'We thank the reviewer for the important point on FAN cost-benefit and physiological interpretability.',
    '',
    '(1) FAN vs MLP FFN (paired segment-level):',
    f'- Mean ΔRMSE (Dual-Transformer): {mean_d_rmse:.6f}',
    f'- Mean ΔSNRi dB (Dual-Transformer): {mean_d_snri:.6f}',
    f'- Dual better fraction: RMSE={frac_rmse:.3f}, SNRi={frac_snri:.3f}',
    '',
    '(2) Cost side:',
    f'- Parameter ratio (Dual / Transformer): {param_ratio:.2f}x',
    '',
    '(3) Harmonic-basis evidence:',
    '- Added FAN periodic-branch (`input_linear_p`) analysis and physiological band-level checks.',
    '',
    'We will revise the manuscript to present FAN gains with paired effect size + consistency + complexity context,',
    'instead of relying on intuition alone.'
]
print('
'.join(lines))
